In [1]:
import sys
import os

# Add project root to Python path
sys.path.append(os.path.abspath(".."))


In [2]:
# ============================================
# 01 — DATA UNDERSTANDING NOTEBOOK
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_dataset
from src.eda.statistics import (
    summarize_dataset,
    missing_values_table,
    unique_values_table,
    detect_outliers_iqr
)
from src.eda.visualization import (
    plot_histograms,
    plot_boxplots,
    plot_categorical_counts,
    plot_time_series,
    plot_transactions_per_account,
)
from src.eda.correlation_analysis import (
    compute_correlations,
    plot_correlation_heatmap
)

# --------------------------------------------
# 1. Load Raw Data
# --------------------------------------------

df = load_dataset("../data/raw/bank_transactions_data_2.csv")
df.head()


,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39


In [3]:
# --------------------------------------------
# 2. Dataset Overview
# --------------------------------------------

summary = summarize_dataset(df)
missing = missing_values_table(df)
unique_vals = unique_values_table(df)

summary, missing, unique_vals


(                     column    dtype  missing_pct  unique_values
 0             TransactionID   object          0.0           2512
 1                 AccountID   object          0.0            495
 2         TransactionAmount  float64          0.0           2455
 3           TransactionDate   object          0.0           2512
 4           TransactionType   object          0.0              2
 5                  Location   object          0.0             43
 6                  DeviceID   object          0.0            681
 7                IP Address   object          0.0            592
 8                MerchantID   object          0.0            100
 9                   Channel   object          0.0              3
 10              CustomerAge    int64          0.0             63
 11       CustomerOccupation   object          0.0              4
 12      TransactionDuration    int64          0.0            288
 13            LoginAttempts    int64          0.0              5
 14       

In [4]:
# Save summary tables
summary.to_csv("../reports/tables/data_summary.csv", index=False)
missing.to_csv("../reports/tables/missing_values.csv", index=False)
unique_vals.to_csv("../reports/tables/unique_counts.csv", index=False)


In [5]:
# --------------------------------------------
# 3. Univariate Analysis
# --------------------------------------------

numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

plot_histograms(df, numeric_cols, save_path="../reports/figures/data_understanding/")
plot_boxplots(df, numeric_cols, save_path="../reports/figures/data_understanding/")
plot_categorical_counts(df, categorical_cols, save_path="../reports/figures/data_understanding/")


In [25]:
# --------------------------------------------
# 4. Outlier Detection
# --------------------------------------------

outlier_report = detect_outliers_iqr(df, numeric_cols)
outlier_report.head()


,column,outliers,lower_bound,upper_bound
0,TransactionAmount,113,-417.07875,913.49125
1,CustomerAge,0,-21.00000,107.00000
2,TransactionDuration,0,-84.00000,308.00000
3,LoginAttempts,122,1.00000,1.00000
4,AccountBalance,0,-7757.30500,16940.49500


In [28]:
# --------------------------------------------
# 5. Time-Based Analysis (if timestamp exists)
# --------------------------------------------

if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    plot_time_series(df, time_col="timestamp", value_col="amount",
                     save_path="../reports/figures/data_understanding/")


In [30]:
# --------------------------------------------
# 6. Correlation Analysis
# --------------------------------------------

corr_matrix = compute_correlations(df[numeric_cols])
plot_correlation_heatmap(corr_matrix, save_path="../reports/figures/data_understanding/")


In [3]:
plot_transactions_per_account(
    df,
    save_path="../reports/figures/data_understanding/"
)

In [4]:
# --------------------------------------------
# 7. Initial Fraud Hypotheses
# --------------------------------------------

print("""
Potential Fraud Indicators:
- High‑value transactions (>900) are potential fraud indicators.
- Multiple login attempts (2–5) strongly suggest account takeover attempts.
- Accounts with unusually high transaction frequency may be mule or laundering accounts.
- High‑frequency IPs/Devices may be shared fraud infrastructure.
- Transactions that consume a large portion of account balance may indicate cash‑out fraud.
""")



Potential Fraud Indicators:
- High‑value transactions (>900) are potential fraud indicators.
- Multiple login attempts (2–5) strongly suggest account takeover attempts.
- Accounts with unusually high transaction frequency may be mule or laundering accounts.
- High‑frequency IPs/Devices may be shared fraud infrastructure.
- Transactions that consume a large portion of account balance may indicate cash‑out fraud.



In [5]:
# --------------------------------------------
# 8. Summary
# --------------------------------------------

print("Phase 1 completed: Data Understanding finished successfully.")


Phase 1 completed: Data Understanding finished successfully.
